In [29]:
import numpy as np
import torch
import torch as th
import pathlib
from pathlib import Path

In [2]:
best_weight_paths = [
    "/data0/shaojiangyi/pprogo-flg-2/models/cc_20251228_103226/hgat_esm2_inductivecc_v3_bestBothStrict.pt",
    "/data0/shaojiangyi/pprogo-flg-2/models/mf_20251226_183932/hgat_esm2_inductivemf_v3_bestBothStrict.pt",
    "/data0/shaojiangyi/pprogo-flg-2/models/bp_20251226_130340/hgat_esm2_inductivebp_v3_bestBothStrict.pt"
]

best_ckpts = [torch.load(p) for p in best_weight_paths]

/tmp/ipykernel_452530/2561103154.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_ckpts = [torch.load(p) for p in best_weight_paths]


In [4]:
best_states = [pt["ema_model"] for pt in best_ckpts]

In [8]:
go_weights = [st["cls_head.weight"] for st in best_states]

In [23]:
[print(m.shape) for m in go_weights]

torch.Size([2852, 256])
torch.Size([6786, 256])
torch.Size([21677, 256])


[None, None, None]

In [10]:
best_pred_paths = [
    "/data0/shaojiangyi/pprogo-flg-2/models/cc_20251228_103226/test/inductive_HGAT_ESM2_cc_v3_bestBothStrict.npz",
    "/data0/shaojiangyi/pprogo-flg-2/models/mf_20251226_183932/test/inductive_HGAT_ESM2_mf_v3_bestBothStrict.npz",
    "/data0/shaojiangyi/pprogo-flg-2/models/bp_20251226_130340/test/inductive_HGAT_ESM2_bp_v3_bestBothStrict.npz"
]

In [11]:
best_preds = [np.load(p) for p in best_pred_paths]

In [15]:
list(best_preds[0].keys())

['protein_names', 'protein_ids', 'predictions', 'labels', 'go_terms', 'split']

In [16]:
goname_lst = [m["go_terms"] for m in best_preds]

In [19]:
union_term_paths = [
    "/data0/shaojiangyi/pprogo-flg-2/results/union_space_preds_only1/cc/test/union_go_terms.npy",
    "/data0/shaojiangyi/pprogo-flg-2/results/union_space_preds_only1/mf/test/union_go_terms.npy",
    "/data0/shaojiangyi/pprogo-flg-2/results/union_space_preds_only1/bp/test/union_go_terms.npy",
]

In [21]:
union_term_lst = [np.load(p, allow_pickle=True) for p in union_term_paths]

In [22]:
def align_labels(predicted_results, o1, o2, fill_value=0.0):
    """
    Align predicted results to match new label order, handling missing labels.
    
    Args:
        predicted_results: numpy array of shape (N, len(o1))
        o1: list - original label order
        o2: list - desired label order (can have different labels/length)
        fill_value: value to use for labels in o2 that don't exist in o1
    
    Returns:
        numpy array of shape (N, len(o2)) - aligned predictions
    """
    N = predicted_results.shape[0]
    M_new = len(o2)
    
    # Initialize new result array
    aligned_results = np.full((N, M_new), fill_value, dtype=predicted_results.dtype)
    
    # Create mapping from label to column index in o1
    o1_label_to_idx = {label: idx for idx, label in enumerate(o1)}
    
    # Fill the aligned results
    for new_idx, label in enumerate(o2):
        if label in o1_label_to_idx:
            old_idx = o1_label_to_idx[label]
            aligned_results[:, new_idx] = predicted_results[:, old_idx]
        # If label not in o1, keep the fill_value (already set)
    
    return aligned_results

In [26]:
go_aligned = []
for i in range(len(go_weights)):
    go_aligned.append(np.transpose(align_labels(np.transpose(go_weights[i].detach().cpu().numpy()), 
                                                goname_lst[i], union_term_lst[i])))

In [28]:
[print(m.shape) for m in go_aligned]

(2881, 256)
(6860, 256)
(21822, 256)


[None, None, None]

In [30]:
ns = ["cc", "mf", "bp"]

In [32]:
saving_dir = Path("/data0/shaojiangyi/pprogo-flg-2/data/")
for i, go_m in enumerate(go_aligned):
    tmp_dir = saving_dir / f"{ns[i]}" / "hgat_esm2_inductive_features"
    if not tmp_dir.exists():
        tmp_dir.mkdir(parents=True)
    saving_path = tmp_dir / "go_features.npy"
    np.save(saving_path, go_weights[i].detach().cpu().numpy())
    saving_path = tmp_dir / "union_go_features.npy"
    np.save(saving_path, go_m)